## Part 4.1: Initialise PostgreSQL Schemas, Tables, Constraints, and Load Session

### Objective

Prepare the PostgreSQL database structures required for controlled Cyclistic data loading.

Parts 1–3 validated the environment, established the approved schema contract, and completed the data-quality audit. Part 4 begins database processing.

This section creates database structures only. It does not load CSV records into staging or quarantine.

### Database Architecture

The initial database-processing layer uses three PostgreSQL schemas:

```text
stg
quarantine
audit
```

Their responsibilities are:

| Schema       | Responsibility                                                                                           |
| ------------ | -------------------------------------------------------------------------------------------------------- |
| `stg`        | Stores source records that pass failure-severity quality controls and remain eligible for transformation |
| `quarantine` | Stores source records that violate failure-severity rules                                                |
| `audit`      | Stores database-load execution history and reconciliation information                                    |

PostgreSQL schemas must exist before schema-qualified tables can be created.

The pipeline therefore creates the schemas explicitly:

```sql
CREATE SCHEMA IF NOT EXISTS stg;
CREATE SCHEMA IF NOT EXISTS quarantine;
CREATE SCHEMA IF NOT EXISTS audit;
```

PostgreSQL and Azure Database for PostgreSQL do not automatically convert:

```text
stg.cyclistic_rides
```

into:

```text
stg_cyclistic_rides
```

The first form refers to table `cyclistic_rides` inside schema `stg`. The second form is a separate table name in the current schema.

### Staging Table

`stg.cyclistic_rides` stores records eligible for downstream processing.

It contains the approved Cyclistic source fields:

* `ride_id`;
* `rideable_type`;
* `started_at`;
* `ended_at`;
* station names and identifiers;
* start and end coordinates;
* `member_casual`.

It also adds technical lineage fields:

* `load_session_id`;
* `quality_audit_session_id`;
* `source_file`;
* `source_row_number`;
* `source_record_key`;
* `ride_duration_seconds`;
* `has_quality_warning`;
* `warning_rule_ids`;
* `loaded_at`.

The source record key uniquely identifies a physical source row within one load session.

### Quarantine Table

`quarantine.cyclistic_rides` preserves records that violate failure-severity quality rules.

It stores:

* the original source values;
* source filename and row number;
* database-load and quality-audit session IDs;
* failed rule IDs;
* quarantine reason;
* optional warning rule IDs;
* quarantine timestamp.

The initial expected quarantine condition is:

```text
DQ007_END_AFTER_START
```

for the 29 records in `202511-divvy-tripdata.csv` whose ending timestamp is not later than their starting timestamp.

No source record is deleted or corrected in this section.

### Pipeline Job Log

`audit.pipeline_job_log` records one row for each database-load execution.

The initial record is inserted with status:

```text
INITIALISED
```

Later Part 4 sections will update the same record through states such as:

```text
LOADING
VALIDATING
COMPLETED
FAILED
```

The table records:

* load session ID;
* related quality-audit session ID;
* schema and ruleset versions;
* source file and row counts;
* staging and quarantine counts;
* rejected and warning counts;
* execution timestamps;
* error information;
* execution metadata.

### Transaction Policy

Schema creation, table creation, index creation, validation, and load-session registration are executed in a PostgreSQL transaction.

If a database operation fails:

* the transaction is rolled back;
* no partial job-log record is committed;
* the failure is written to `pipeline_audit.log`;
* the notebook raises the original error.

### Idempotency Policy

The database objects use:

```sql
CREATE SCHEMA IF NOT EXISTS
CREATE TABLE IF NOT EXISTS
CREATE INDEX IF NOT EXISTS
```

This allows the setup cell to be rerun safely.

However, `IF NOT EXISTS` does not prove that an existing table has the correct columns. The section therefore validates required table columns after creation and stops if an incompatible existing table is detected.

### Expected Outcome

* PostgreSQL connectivity is reverified.
* The current database and role are identified.
* Schema-creation permission is confirmed.
* `stg`, `quarantine`, and `audit` exist.
* Required tables and indexes exist.
* Existing table structures contain all required columns.
* A unique database-load session is generated.
* The load session is registered as `INITIALISED`.
* Part 4.2 can begin controlled chunked loading and row routing.


In [ ]:
# ==========================================
# Part 4.1: Initialise PostgreSQL Schemas
# ==========================================

from datetime import datetime
from pathlib import Path
from uuid import uuid4

import psycopg2
from psycopg2 import OperationalError, sql


# ------------------------------------------
# Validate required dependencies
# ------------------------------------------

REQUIRED_PART_4_1_VALUES = {
    "DB_HOST": DB_HOST,
    "DB_NAME": DB_NAME,
    "DB_USER": DB_USER,
    "DB_PASSWORD": DB_PASSWORD,
    "DB_PORT": DB_PORT,
    "DB_SSLMODE": DB_SSLMODE,
    "PIPELINE_AUDIT_PATH": PIPELINE_AUDIT_PATH,
    "MELBOURNE_TIMEZONE": MELBOURNE_TIMEZONE,
}

missing_part_4_1_values = [
    variable_name
    for variable_name, variable_value
    in REQUIRED_PART_4_1_VALUES.items()
    if (
        variable_value is None
        or (
            isinstance(variable_value, str)
            and not variable_value.strip()
        )
    )
]

if missing_part_4_1_values:
    raise RuntimeError(
        "Part 4.1 is missing required value(s): "
        + ", ".join(
            sorted(missing_part_4_1_values)
        )
    )


if not isinstance(
    PIPELINE_AUDIT_PATH,
    Path,
):
    raise TypeError(
        "PIPELINE_AUDIT_PATH must be a pathlib.Path object."
    )


if not PIPELINE_AUDIT_PATH.exists():
    raise FileNotFoundError(
        "Pipeline audit log does not exist:\n"
        f"{PIPELINE_AUDIT_PATH}"
    )


if not PIPELINE_AUDIT_PATH.is_file():
    raise FileExistsError(
        "PIPELINE_AUDIT_PATH exists but is not a file:\n"
        f"{PIPELINE_AUDIT_PATH}"
    )


# ------------------------------------------
# Initialise database-processing session
# ------------------------------------------

# Reuse this identifier throughout Parts 4.1–4.5.
DATABASE_PROCESSING_SESSION_ID = (
    uuid4().hex[:8]
)

DATABASE_PROCESSING_START_TIME = (
    datetime.now(
        MELBOURNE_TIMEZONE
    )
)

database_run_warnings = []
database_run_errors = []


SUPPORTED_DATABASE_LOG_LEVELS = {
    "INFO",
    "WARNING",
    "ERROR",
    "CRITICAL",
}


def log_database_event(
    message: str,
    level: str = "INFO",
) -> str:
    """
    Append one database-processing event to the shared pipeline
    audit log and print it in the notebook.
    """
    normalised_level = level.upper().strip()

    if (
        normalised_level
        not in SUPPORTED_DATABASE_LOG_LEVELS
    ):
        raise ValueError(
            "Unsupported database log level: "
            f"{level}"
        )

    if (
        not isinstance(message, str)
        or not message.strip()
    ):
        raise ValueError(
            "Database log message must be a non-empty string."
        )

    timestamp = datetime.now(
        MELBOURNE_TIMEZONE
    ).isoformat(timespec="seconds")

    log_entry = (
        f"[{timestamp}] "
        f"[DB_RUN:{DATABASE_PROCESSING_SESSION_ID}] "
        f"[{normalised_level}] "
        f"{message.strip()}"
    )

    with PIPELINE_AUDIT_PATH.open(
        mode="a",
        encoding="utf-8",
    ) as audit_file:
        audit_file.write(
            log_entry + "\n"
        )

    print(log_entry)

    event_record = {
        "timestamp": timestamp,
        "database_processing_session_id": (
            DATABASE_PROCESSING_SESSION_ID
        ),
        "level": normalised_level,
        "message": message.strip(),
    }

    if normalised_level == "WARNING":
        database_run_warnings.append(
            event_record
        )

    elif normalised_level in {
        "ERROR",
        "CRITICAL",
    }:
        database_run_errors.append(
            event_record
        )

    return log_entry


# ------------------------------------------
# Define required PostgreSQL schemas
# ------------------------------------------

REQUIRED_DATABASE_SCHEMAS = (
    "stg",
    "quarantine",
    "audit",
)


# ------------------------------------------
# Open transaction and initialise schemas
# ------------------------------------------

connection = None
cursor = None

schemas_existing_before = []
schemas_created_this_run = []
verified_schemas = []


log_database_event(
    message=(
        "DATABASE_PROCESSING_SESSION_START: "
        "PostgreSQL database processing initialised."
    ),
    level="INFO",
)


try:
    connection = psycopg2.connect(
        host=DB_HOST,
        dbname=DB_NAME,
        user=DB_USER,
        password=DB_PASSWORD,
        port=DB_PORT,
        sslmode=DB_SSLMODE,
        connect_timeout=10,
    )

    connection.autocommit = False
    cursor = connection.cursor()


    # --------------------------------------
    # Confirm connection identity
    # --------------------------------------

    cursor.execute(
        """
        SELECT
            current_database(),
            current_user;
        """
    )

    (
        connected_database,
        connected_user,
    ) = cursor.fetchone()


    if connected_database != DB_NAME:
        raise RuntimeError(
            "Connected database does not match DB_NAME. "
            f"Expected={DB_NAME}; "
            f"Observed={connected_database}."
        )


    log_database_event(
        message=(
            "DATABASE_CONNECTION_OPENED: "
            f"database={connected_database}; "
            f"user={connected_user}; "
            f"host={DB_HOST}; "
            f"port={DB_PORT}."
        ),
        level="INFO",
    )


    # --------------------------------------
    # Inspect schemas before initialisation
    # --------------------------------------

    cursor.execute(
        """
        SELECT schema_name
        FROM information_schema.schemata
        WHERE schema_name = ANY(%s)
        ORDER BY schema_name;
        """,
        (list(REQUIRED_DATABASE_SCHEMAS),),
    )

    schemas_existing_before = [
        row[0]
        for row in cursor.fetchall()
    ]


    schemas_missing_before = [
        schema_name
        for schema_name
        in REQUIRED_DATABASE_SCHEMAS
        if schema_name
        not in schemas_existing_before
    ]


    # --------------------------------------
    # Validate database CREATE privilege
    # --------------------------------------

    cursor.execute(
        """
        SELECT has_database_privilege(
            current_user,
            current_database(),
            'CREATE'
        );
        """
    )

    can_create_schema = bool(
        cursor.fetchone()[0]
    )


    if (
        schemas_missing_before
        and not can_create_schema
    ):
        raise PermissionError(
            "The connected PostgreSQL role cannot create "
            "schemas in the current database. "
            "Missing schema(s): "
            + ", ".join(
                schemas_missing_before
            )
        )


    # --------------------------------------
    # Create schemas idempotently
    # --------------------------------------

    for schema_name in (
        REQUIRED_DATABASE_SCHEMAS
    ):
        create_schema_statement = (
            sql.SQL(
                "CREATE SCHEMA IF NOT EXISTS {}"
            ).format(
                sql.Identifier(schema_name)
            )
        )

        cursor.execute(
            create_schema_statement
        )


        if schema_name in schemas_existing_before:
            log_database_event(
                message=(
                    "DATABASE_SCHEMA_FOUND: "
                    f"schema={schema_name}."
                ),
                level="INFO",
            )

        else:
            schemas_created_this_run.append(
                schema_name
            )

            log_database_event(
                message=(
                    "DATABASE_SCHEMA_CREATED: "
                    f"schema={schema_name}."
                ),
                level="INFO",
            )


    # --------------------------------------
    # Verify schema existence and ownership
    # --------------------------------------

    cursor.execute(
        """
        SELECT
            namespace.nspname AS schema_name,
            owner_role.rolname AS schema_owner
        FROM pg_namespace AS namespace
        INNER JOIN pg_roles AS owner_role
            ON owner_role.oid = namespace.nspowner
        WHERE namespace.nspname = ANY(%s)
        ORDER BY namespace.nspname;
        """,
        (list(REQUIRED_DATABASE_SCHEMAS),),
    )

    verified_schema_rows = (
        cursor.fetchall()
    )

    verified_schema_metadata = {
        schema_name: {
            "owner": schema_owner,
        }
        for (
            schema_name,
            schema_owner,
        ) in verified_schema_rows
    }


    missing_after_initialisation = [
        schema_name
        for schema_name
        in REQUIRED_DATABASE_SCHEMAS
        if schema_name
        not in verified_schema_metadata
    ]


    if missing_after_initialisation:
        raise RuntimeError(
            "Required schema verification failed. "
            "Missing schema(s): "
            + ", ".join(
                missing_after_initialisation
            )
        )


    # --------------------------------------
    # Verify schema privileges
    # --------------------------------------

    for schema_name in (
        REQUIRED_DATABASE_SCHEMAS
    ):
        cursor.execute(
            """
            SELECT
                has_schema_privilege(
                    current_user,
                    %s,
                    'USAGE'
                ),
                has_schema_privilege(
                    current_user,
                    %s,
                    'CREATE'
                );
            """,
            (
                schema_name,
                schema_name,
            ),
        )

        (
            has_usage_privilege,
            has_create_privilege,
        ) = cursor.fetchone()


        if not has_usage_privilege:
            raise PermissionError(
                "The connected PostgreSQL role does not have "
                f"USAGE privilege on schema '{schema_name}'."
            )


        if not has_create_privilege:
            raise PermissionError(
                "The connected PostgreSQL role does not have "
                f"CREATE privilege on schema '{schema_name}'."
            )


        verified_schemas.append(
            {
                "schema_name": schema_name,
                "owner": (
                    verified_schema_metadata[
                        schema_name
                    ]["owner"]
                ),
                "usage_privilege": bool(
                    has_usage_privilege
                ),
                "create_privilege": bool(
                    has_create_privilege
                ),
            }
        )

        log_database_event(
            message=(
                "DATABASE_SCHEMA_VERIFIED: "
                f"schema={schema_name}; "
                f"owner="
                f"{verified_schema_metadata[schema_name]['owner']}; "
                "usage_privilege=True; "
                "create_privilege=True."
            ),
            level="INFO",
        )


    # --------------------------------------
    # Commit schema initialisation
    # --------------------------------------

    connection.commit()

    log_database_event(
        message=(
            "DATABASE_SCHEMA_INITIALISATION_COMMITTED: "
            f"created={len(schemas_created_this_run)}; "
            f"existing={len(schemas_existing_before)}; "
            f"verified={len(verified_schemas)}."
        ),
        level="INFO",
    )


except OperationalError as error:
    if connection is not None:
        connection.rollback()

    log_database_event(
        message=(
            "DATABASE_SCHEMA_INITIALISATION_FAILED: "
            "Unable to connect to PostgreSQL. "
            f"error={error}"
        ),
        level="CRITICAL",
    )

    raise ConnectionError(
        "Unable to open the PostgreSQL connection required "
        "for Part 4.1."
    ) from error


except (
    PermissionError,
    RuntimeError,
    psycopg2.DatabaseError,
) as error:
    if connection is not None:
        connection.rollback()

    log_database_event(
        message=(
            "DATABASE_SCHEMA_INITIALISATION_ROLLED_BACK: "
            f"error_type={type(error).__name__}; "
            f"error={error}"
        ),
        level="CRITICAL",
    )

    raise


finally:
    if cursor is not None:
        cursor.close()
        print("✓ Database cursor closed.")

    if connection is not None:
        connection.close()
        print("✓ Database connection closed.")


# ------------------------------------------
# Build Part 4.1 summary
# ------------------------------------------

DATABASE_SCHEMA_INITIALISATION_SUMMARY = {
    "database_processing_session_id": (
        DATABASE_PROCESSING_SESSION_ID
    ),
    "database": connected_database,
    "connected_user": connected_user,
    "required_schemas": list(
        REQUIRED_DATABASE_SCHEMAS
    ),
    "schemas_existing_before": list(
        schemas_existing_before
    ),
    "schemas_created_this_run": list(
        schemas_created_this_run
    ),
    "verified_schemas": list(
        verified_schemas
    ),
    "status": "SUCCESS",
    "completed_at": datetime.now(
        MELBOURNE_TIMEZONE
    ).isoformat(timespec="seconds"),
}


# ------------------------------------------
# Display final summary
# ------------------------------------------

print(
    "\n========== PostgreSQL Schema Initialisation "
    "==========\n"
)

print(
    f"Database-processing session : "
    f"{DATABASE_PROCESSING_SESSION_ID}"
)
print(
    f"Database                    : "
    f"{connected_database}"
)
print(
    f"Connected role              : "
    f"{connected_user}"
)
print(
    f"Required schemas            : "
    f"{len(REQUIRED_DATABASE_SCHEMAS)}"
)
print(
    f"Schemas created             : "
    f"{len(schemas_created_this_run)}"
)
print(
    f"Schemas already existing    : "
    f"{len(schemas_existing_before)}"
)
print(
    f"Schemas verified            : "
    f"{len(verified_schemas)}"
)


print("\nSchema details:")

for schema_record in verified_schemas:
    print(
        f"  {schema_record['schema_name']:<12} "
        f"owner={schema_record['owner']:<20} "
        "USAGE=True CREATE=True"
    )


print("\n" + "=" * 75)
print("✓ Required PostgreSQL schemas exist.")
print("✓ Existing schemas were preserved.")
print("✓ Missing schemas were created idempotently.")
print("✓ Schema permissions verified.")
print("✓ Schema initialisation transaction committed.")
print("✓ Part 4.1 completed successfully.")
print("=" * 75)

## Part 4.2: Provision and Validate PostgreSQL Tables

### Objective

Create and validate the PostgreSQL tables required for controlled Cyclistic data loading.

Part 4.1 established the database namespaces:

```text
stg
quarantine
audit
```

Part 4.2 provisions the following tables:

```text
stg.cyclistic_rides
quarantine.cyclistic_rides
audit.pipeline_job_log
```

This section creates database infrastructure only. It does not read CSV files or insert source records.

### Approved Schema Dependency

The staging table must be based on the approved schema stored in:

```text
master_schema.json
```

Before table provisioning begins, this section reloads and verifies:

* `active_schema_status` is `APPROVED`;
* `active_schema` is not empty;
* `active_schema_version` is populated;
* all approved SQL types match a controlled type allowlist.

The approved schema is never inferred or modified during Part 4.2.

### Staging Table

The staging table contains the approved Cyclistic columns plus operational lineage fields:

```text
staging_record_id
source_file
source_row_number
pipeline_session_id
quality_audit_session_id
loaded_at
```

`staging_record_id` is the technical primary key.

`ride_id` receives a unique index as a defensive database control because the approved quality contract requires it to be unique across the source dataset.

The lineage fields allow each loaded row to be traced back to:

* its source CSV;
* its source row number;
* the database-processing session;
* the quality-audit session.

### Quarantine Table

The quarantine table stores records that violate failure-severity quality rules.

It contains:

```text
quarantine_id
ride_id
failed_rule
reason
source_file
source_row_number
raw_record
pipeline_session_id
quality_audit_session_id
created_at
```

`raw_record` uses PostgreSQL `JSONB` so the complete source record can be preserved without requiring the quarantine table to duplicate every approved schema column.

One source row may eventually produce more than one quarantine finding when multiple failure rules are violated.

### Pipeline Job Log

`audit.pipeline_job_log` records each source-file processing attempt.

It includes:

* job ID;
* pipeline session ID;
* quality-audit session ID;
* source filename;
* SHA-256 file checksum;
* status;
* start and completion timestamps;
* rows read;
* rows loaded;
* rows quarantined;
* duration;
* approved schema version;
* quality-ruleset version;
* error details.

A partial unique index is created for successful loads:

```text
source_file + file_checksum
```

This supports the agreed idempotency policy:

```text
If the same file and checksum already completed successfully,
skip it unless a future force-reload option is enabled.
```

Failed attempts may still be recorded multiple times.

### Existing-Table Policy

`CREATE TABLE IF NOT EXISTS` prevents accidental replacement of existing tables, but it does not verify their definitions.

After provisioning, Part 4.2 inspects PostgreSQL metadata and validates:

* required columns;
* SQL data types;
* nullability;
* primary keys;
* named indexes;
* required constraints.

If an existing table differs from the expected contract, the transaction is rolled back and the table is not silently altered.

Schema migration and controlled table alteration should be handled explicitly rather than hidden inside normal provisioning.

### Transaction Policy

All table and index provisioning occurs in one transaction.

```text
Open connection
↓
Reload approved schema
↓
Create missing tables
↓
Create missing indexes
↓
Inspect live table metadata
↓
Validate columns and constraints
↓
Commit
```

If validation fails:

```text
Rollback
↓
Log the mismatch
↓
Close the connection
```

### Expected Outcome

* All three required tables exist.
* The staging table matches the approved schema plus lineage columns.
* The quarantine table supports complete row preservation and failure lineage.
* The job-log table supports file-level execution history and idempotency.
* Primary keys, constraints, and indexes are validated.
* No source records are loaded.
* Part 4 is ready for load-session initialisation and idempotency checks.


In [ ]:
# ==========================================
# Part 4.2: Provision and Validate
# PostgreSQL Tables
# ==========================================

import json
import re
from copy import deepcopy
from datetime import datetime
from pathlib import Path
from typing import Any

import psycopg2
from psycopg2 import OperationalError, sql


# ------------------------------------------
# Validate required dependencies
# ------------------------------------------

REQUIRED_PART_4_2_VALUES = {
    "DATABASE_PROCESSING_SESSION_ID": (
        DATABASE_PROCESSING_SESSION_ID
    ),
    "DB_HOST": DB_HOST,
    "DB_NAME": DB_NAME,
    "DB_USER": DB_USER,
    "DB_PASSWORD": DB_PASSWORD,
    "DB_PORT": DB_PORT,
    "DB_SSLMODE": DB_SSLMODE,
    "MASTER_SCHEMA_PATH": MASTER_SCHEMA_PATH,
    "QUALITY_RULES_PATH": QUALITY_RULES_PATH,
    "MELBOURNE_TIMEZONE": MELBOURNE_TIMEZONE,
}

missing_part_4_2_values = [
    variable_name
    for variable_name, variable_value
    in REQUIRED_PART_4_2_VALUES.items()
    if (
        variable_value is None
        or (
            isinstance(variable_value, str)
            and not variable_value.strip()
        )
    )
]

if missing_part_4_2_values:
    raise RuntimeError(
        "Part 4.2 is missing required value(s): "
        + ", ".join(
            sorted(missing_part_4_2_values)
        )
    )


for path_name in (
    "MASTER_SCHEMA_PATH",
    "QUALITY_RULES_PATH",
):
    path_value = REQUIRED_PART_4_2_VALUES[
        path_name
    ]

    if not isinstance(path_value, Path):
        raise TypeError(
            f"{path_name} must be a pathlib.Path object."
        )

    if not path_value.exists():
        raise FileNotFoundError(
            f"{path_name} does not exist:\n"
            f"{path_value}"
        )

    if not path_value.is_file():
        raise FileExistsError(
            f"{path_name} exists but is not a file:\n"
            f"{path_value}"
        )


# ------------------------------------------
# Reload approved schema and quality metadata
# ------------------------------------------

def load_required_json_object(
    file_path: Path,
    object_name: str,
) -> dict:
    """
    Load a required JSON object without using stale notebook state.
    """
    try:
        with file_path.open(
            mode="r",
            encoding="utf-8",
        ) as json_file:
            loaded_data = json.load(json_file)

    except json.JSONDecodeError as error:
        raise ValueError(
            f"{object_name} contains invalid JSON."
        ) from error

    if not isinstance(loaded_data, dict):
        raise TypeError(
            f"{object_name} must contain a JSON object."
        )

    return loaded_data


database_master_registry = (
    load_required_json_object(
        file_path=MASTER_SCHEMA_PATH,
        object_name="master_schema.json",
    )
)

database_quality_rules = (
    load_required_json_object(
        file_path=QUALITY_RULES_PATH,
        object_name="quality_rules.json",
    )
)


DATABASE_ACTIVE_SCHEMA = (
    database_master_registry.get(
        "active_schema",
        {},
    )
)

DATABASE_ACTIVE_SCHEMA_STATUS = (
    database_master_registry.get(
        "active_schema_status"
    )
)

DATABASE_ACTIVE_SCHEMA_VERSION = (
    database_master_registry.get(
        "active_schema_version"
    )
)

DATABASE_QUALITY_RULESET_VERSION = (
    database_quality_rules.get(
        "metadata",
        {},
    ).get(
        "ruleset_version"
    )
)


if DATABASE_ACTIVE_SCHEMA_STATUS != "APPROVED":
    raise RuntimeError(
        "Part 4.2 requires active_schema_status=APPROVED."
    )


if (
    not isinstance(
        DATABASE_ACTIVE_SCHEMA,
        dict,
    )
    or not DATABASE_ACTIVE_SCHEMA
):
    raise RuntimeError(
        "Part 4.2 requires a non-empty active_schema."
    )


if (
    not isinstance(
        DATABASE_ACTIVE_SCHEMA_VERSION,
        str,
    )
    or not DATABASE_ACTIVE_SCHEMA_VERSION.strip()
):
    raise RuntimeError(
        "Part 4.2 requires a valid active_schema_version."
    )


if (
    not isinstance(
        DATABASE_QUALITY_RULESET_VERSION,
        str,
    )
    or not DATABASE_QUALITY_RULESET_VERSION.strip()
):
    raise RuntimeError(
        "Part 4.2 requires a valid quality ruleset version."
    )


# ------------------------------------------
# Validate approved SQL type definitions
# ------------------------------------------

ALLOWED_SQL_TYPE_PATTERNS = (
    re.compile(
        r"^VARCHAR\([1-9][0-9]*\)$",
        re.IGNORECASE,
    ),
    re.compile(
        r"^NUMERIC\([1-9][0-9]*,\s*[0-9]+\)$",
        re.IGNORECASE,
    ),
    re.compile(
        r"^(SMALLINT|INTEGER|BIGINT)$",
        re.IGNORECASE,
    ),
    re.compile(
        r"^(REAL|DOUBLE PRECISION)$",
        re.IGNORECASE,
    ),
    re.compile(
        r"^(BOOLEAN|DATE|TEXT|UUID|JSONB)$",
        re.IGNORECASE,
    ),
    re.compile(
        r"^TIMESTAMP"
        r"( WITHOUT TIME ZONE| WITH TIME ZONE)?$",
        re.IGNORECASE,
    ),
)


def normalise_declared_sql_type(
    sql_type_value: str,
) -> str:
    """
    Normalise whitespace and capitalisation in an approved SQL type.
    """
    if not isinstance(
        sql_type_value,
        str,
    ):
        raise TypeError(
            "Approved SQL type values must be strings."
        )

    normalised_type = re.sub(
        r"\s+",
        " ",
        sql_type_value.strip().upper(),
    )

    if not any(
        pattern.fullmatch(normalised_type)
        for pattern in ALLOWED_SQL_TYPE_PATTERNS
    ):
        raise ValueError(
            "Unsupported or unsafe approved SQL type: "
            f"{sql_type_value}"
        )

    return normalised_type


NORMALISED_ACTIVE_SCHEMA = {
    column_name: normalise_declared_sql_type(
        sql_type_value
    )
    for column_name, sql_type_value
    in DATABASE_ACTIVE_SCHEMA.items()
}


# ------------------------------------------
# Protect reserved lineage column names
# ------------------------------------------

STAGING_LINEAGE_COLUMNS = {
    "staging_record_id",
    "source_file",
    "source_row_number",
    "pipeline_session_id",
    "quality_audit_session_id",
    "loaded_at",
}


lineage_name_conflicts = (
    STAGING_LINEAGE_COLUMNS
    & set(NORMALISED_ACTIVE_SCHEMA)
)

if lineage_name_conflicts:
    raise ValueError(
        "Approved schema conflicts with staging lineage column(s): "
        + ", ".join(
            sorted(lineage_name_conflicts)
        )
    )


if "ride_id" not in NORMALISED_ACTIVE_SCHEMA:
    raise ValueError(
        "Approved schema must contain ride_id."
    )


# ------------------------------------------
# Define expected table contracts
# ------------------------------------------

def column_contract(
    sql_type: str,
    nullable: bool,
) -> dict:
    return {
        "sql_type": sql_type,
        "nullable": nullable,
    }


EXPECTED_STAGING_COLUMNS = {
    "staging_record_id": column_contract(
        "UUID",
        False,
    ),
    **{
        column_name: column_contract(
            sql_type_value,
            False
            if column_name == "ride_id"
            else True,
        )
        for column_name, sql_type_value
        in NORMALISED_ACTIVE_SCHEMA.items()
    },
    "source_file": column_contract(
        "VARCHAR(255)",
        False,
    ),
    "source_row_number": column_contract(
        "BIGINT",
        False,
    ),
    "pipeline_session_id": column_contract(
        "VARCHAR(64)",
        False,
    ),
    "quality_audit_session_id": column_contract(
        "VARCHAR(64)",
        False,
    ),
    "loaded_at": column_contract(
        "TIMESTAMP WITH TIME ZONE",
        False,
    ),
}


EXPECTED_QUARANTINE_COLUMNS = {
    "quarantine_id": column_contract(
        "UUID",
        False,
    ),
    "ride_id": column_contract(
        "VARCHAR(255)",
        True,
    ),
    "failed_rule": column_contract(
        "VARCHAR(100)",
        False,
    ),
    "reason": column_contract(
        "TEXT",
        False,
    ),
    "source_file": column_contract(
        "VARCHAR(255)",
        False,
    ),
    "source_row_number": column_contract(
        "BIGINT",
        False,
    ),
    "raw_record": column_contract(
        "JSONB",
        False,
    ),
    "pipeline_session_id": column_contract(
        "VARCHAR(64)",
        False,
    ),
    "quality_audit_session_id": column_contract(
        "VARCHAR(64)",
        False,
    ),
    "created_at": column_contract(
        "TIMESTAMP WITH TIME ZONE",
        False,
    ),
}


EXPECTED_JOB_LOG_COLUMNS = {
    "job_id": column_contract(
        "UUID",
        False,
    ),
    "pipeline_session_id": column_contract(
        "VARCHAR(64)",
        False,
    ),
    "quality_audit_session_id": column_contract(
        "VARCHAR(64)",
        False,
    ),
    "source_file": column_contract(
        "VARCHAR(255)",
        False,
    ),
    "file_checksum": column_contract(
        "VARCHAR(64)",
        False,
    ),
    "status": column_contract(
        "VARCHAR(30)",
        False,
    ),
    "started_at": column_contract(
        "TIMESTAMP WITH TIME ZONE",
        False,
    ),
    "completed_at": column_contract(
        "TIMESTAMP WITH TIME ZONE",
        True,
    ),
    "rows_read": column_contract(
        "BIGINT",
        False,
    ),
    "rows_loaded": column_contract(
        "BIGINT",
        False,
    ),
    "rows_quarantined": column_contract(
        "BIGINT",
        False,
    ),
    "duration_seconds": column_contract(
        "NUMERIC(18, 3)",
        True,
    ),
    "schema_version": column_contract(
        "VARCHAR(50)",
        False,
    ),
    "quality_rules_version": column_contract(
        "VARCHAR(50)",
        False,
    ),
    "error_message": column_contract(
        "TEXT",
        True,
    ),
    "created_at": column_contract(
        "TIMESTAMP WITH TIME ZONE",
        False,
    ),
    "updated_at": column_contract(
        "TIMESTAMP WITH TIME ZONE",
        False,
    ),
}


EXPECTED_TABLE_CONTRACTS = {
    ("audit", "pipeline_job_log"): {
        "columns": EXPECTED_JOB_LOG_COLUMNS,
        "primary_key": ["job_id"],
        "required_constraints": {
            "ck_pipeline_job_log_status",
            "ck_pipeline_job_log_counts",
        },
        "required_indexes": {
            "ux_pipeline_job_log_success_file",
            "ix_pipeline_job_log_session",
            "ix_pipeline_job_log_source_file",
        },
    },
    ("stg", "cyclistic_rides"): {
        "columns": EXPECTED_STAGING_COLUMNS,
        "primary_key": [
            "staging_record_id"
        ],
        "required_constraints": set(),
        "required_indexes": {
            "ux_stg_cyclistic_rides_ride_id",
            "ix_stg_cyclistic_rides_source",
            "ix_stg_cyclistic_rides_session",
        },
    },
    ("quarantine", "cyclistic_rides"): {
        "columns": EXPECTED_QUARANTINE_COLUMNS,
        "primary_key": [
            "quarantine_id"
        ],
        "required_constraints": set(),
        "required_indexes": {
            "ix_quarantine_cyclistic_rides_source",
            "ix_quarantine_cyclistic_rides_rule",
            "ix_quarantine_cyclistic_rides_session",
        },
    },
}


# ------------------------------------------
# PostgreSQL metadata normalisation
# ------------------------------------------

def canonical_database_type(
    data_type: str,
    character_maximum_length: int | None,
    numeric_precision: int | None,
    numeric_scale: int | None,
) -> str:
    """
    Convert information_schema type metadata into a canonical form.
    """
    normalised_data_type = (
        data_type.strip().upper()
    )

    if normalised_data_type in {
        "CHARACTER VARYING",
        "VARCHAR",
    }:
        return (
            f"VARCHAR("
            f"{character_maximum_length}"
            f")"
        )

    if normalised_data_type in {
        "NUMERIC",
        "DECIMAL",
    }:
        return (
            f"NUMERIC("
            f"{numeric_precision}, "
            f"{numeric_scale}"
            f")"
        )

    if normalised_data_type in {
        "TIMESTAMP WITHOUT TIME ZONE",
        "TIMESTAMP WITH TIME ZONE",
    }:
        return normalised_data_type

    return normalised_data_type


def normalise_type_spacing(
    sql_type_value: str,
) -> str:
    """
    Normalise spacing around commas for reliable comparisons.
    """
    normalised_value = re.sub(
        r"\s+",
        " ",
        sql_type_value.strip().upper(),
    )

    normalised_value = re.sub(
        r"\s*,\s*",
        ", ",
        normalised_value,
    )

    if normalised_value == "TIMESTAMP":
        return "TIMESTAMP WITHOUT TIME ZONE"

    return normalised_value


# ------------------------------------------
# Generate staging table SQL safely
# ------------------------------------------

def build_staging_table_statement():
    """
    Build the staging table from the approved schema and lineage
    contract.
    """
    column_definitions = [
        sql.SQL(
            "{} UUID NOT NULL"
        ).format(
            sql.Identifier(
                "staging_record_id"
            )
        )
    ]

    for column_name, sql_type_value in (
        NORMALISED_ACTIVE_SCHEMA.items()
    ):
        nullability_sql = (
            sql.SQL(" NOT NULL")
            if column_name == "ride_id"
            else sql.SQL("")
        )

        column_definitions.append(
            sql.SQL(
                "{} {}{}"
            ).format(
                sql.Identifier(column_name),
                sql.SQL(sql_type_value),
                nullability_sql,
            )
        )

    column_definitions.extend(
        [
            sql.SQL(
                "{} VARCHAR(255) NOT NULL"
            ).format(
                sql.Identifier(
                    "source_file"
                )
            ),
            sql.SQL(
                "{} BIGINT NOT NULL"
            ).format(
                sql.Identifier(
                    "source_row_number"
                )
            ),
            sql.SQL(
                "{} VARCHAR(64) NOT NULL"
            ).format(
                sql.Identifier(
                    "pipeline_session_id"
                )
            ),
            sql.SQL(
                "{} VARCHAR(64) NOT NULL"
            ).format(
                sql.Identifier(
                    "quality_audit_session_id"
                )
            ),
            sql.SQL(
                "{} TIMESTAMPTZ NOT NULL "
                "DEFAULT CURRENT_TIMESTAMP"
            ).format(
                sql.Identifier(
                    "loaded_at"
                )
            ),
            sql.SQL(
                "CONSTRAINT {} PRIMARY KEY ({})"
            ).format(
                sql.Identifier(
                    "pk_stg_cyclistic_rides"
                ),
                sql.Identifier(
                    "staging_record_id"
                ),
            ),
        ]
    )

    return sql.SQL(
        "CREATE TABLE IF NOT EXISTS {}.{} ({})"
    ).format(
        sql.Identifier("stg"),
        sql.Identifier(
            "cyclistic_rides"
        ),
        sql.SQL(", ").join(
            column_definitions
        ),
    )


# ------------------------------------------
# Table validation helpers
# ------------------------------------------

def fetch_table_columns(
    cursor,
    schema_name: str,
    table_name: str,
) -> dict:
    """
    Read live column metadata from information_schema.
    """
    cursor.execute(
        """
        SELECT
            column_name,
            data_type,
            character_maximum_length,
            numeric_precision,
            numeric_scale,
            is_nullable
        FROM information_schema.columns
        WHERE
            table_schema = %s
            AND table_name = %s
        ORDER BY ordinal_position;
        """,
        (
            schema_name,
            table_name,
        ),
    )

    live_columns = {}

    for (
        column_name,
        data_type,
        character_maximum_length,
        numeric_precision,
        numeric_scale,
        is_nullable,
    ) in cursor.fetchall():

        live_columns[column_name] = {
            "sql_type": canonical_database_type(
                data_type=(
                    data_type
                ),
                character_maximum_length=(
                    character_maximum_length
                ),
                numeric_precision=(
                    numeric_precision
                ),
                numeric_scale=(
                    numeric_scale
                ),
            ),
            "nullable": (
                is_nullable == "YES"
            ),
        }

    return live_columns


def fetch_primary_key_columns(
    cursor,
    schema_name: str,
    table_name: str,
) -> list[str]:
    """
    Return ordered primary-key columns.
    """
    cursor.execute(
        """
        SELECT
            attribute.attname
        FROM pg_constraint AS constraint_record
        INNER JOIN pg_class AS table_record
            ON table_record.oid =
               constraint_record.conrelid
        INNER JOIN pg_namespace AS namespace_record
            ON namespace_record.oid =
               table_record.relnamespace
        INNER JOIN unnest(
            constraint_record.conkey
        ) WITH ORDINALITY AS key_column(
            attribute_number,
            key_order
        )
            ON TRUE
        INNER JOIN pg_attribute AS attribute
            ON attribute.attrelid =
               table_record.oid
            AND attribute.attnum =
                key_column.attribute_number
        WHERE
            constraint_record.contype = 'p'
            AND namespace_record.nspname = %s
            AND table_record.relname = %s
        ORDER BY key_column.key_order;
        """,
        (
            schema_name,
            table_name,
        ),
    )

    return [
        row[0]
        for row in cursor.fetchall()
    ]


def fetch_constraint_names(
    cursor,
    schema_name: str,
    table_name: str,
) -> set[str]:
    """
    Return all named constraints attached to a table.
    """
    cursor.execute(
        """
        SELECT constraint_record.conname
        FROM pg_constraint AS constraint_record
        INNER JOIN pg_class AS table_record
            ON table_record.oid =
               constraint_record.conrelid
        INNER JOIN pg_namespace AS namespace_record
            ON namespace_record.oid =
               table_record.relnamespace
        WHERE
            namespace_record.nspname = %s
            AND table_record.relname = %s;
        """,
        (
            schema_name,
            table_name,
        ),
    )

    return {
        row[0]
        for row in cursor.fetchall()
    }


def fetch_index_names(
    cursor,
    schema_name: str,
    table_name: str,
) -> set[str]:
    """
    Return PostgreSQL index names for a table.
    """
    cursor.execute(
        """
        SELECT indexname
        FROM pg_indexes
        WHERE
            schemaname = %s
            AND tablename = %s;
        """,
        (
            schema_name,
            table_name,
        ),
    )

    return {
        row[0]
        for row in cursor.fetchall()
    }


def validate_table_contract(
    cursor,
    schema_name: str,
    table_name: str,
    expected_contract: dict,
) -> dict:
    """
    Validate a live table against its expected structural contract.
    """
    live_columns = fetch_table_columns(
        cursor=cursor,
        schema_name=schema_name,
        table_name=table_name,
    )

    expected_columns = (
        expected_contract["columns"]
    )

    missing_columns = sorted(
        set(expected_columns)
        - set(live_columns)
    )

    unexpected_columns = sorted(
        set(live_columns)
        - set(expected_columns)
    )

    type_mismatches = {}
    nullability_mismatches = {}

    for column_name in (
        set(expected_columns)
        & set(live_columns)
    ):
        expected_type = (
            normalise_type_spacing(
                expected_columns[
                    column_name
                ]["sql_type"]
            )
        )

        observed_type = (
            normalise_type_spacing(
                live_columns[
                    column_name
                ]["sql_type"]
            )
        )

        if expected_type != observed_type:
            type_mismatches[
                column_name
            ] = {
                "expected": expected_type,
                "observed": observed_type,
            }

        expected_nullable = (
            expected_columns[
                column_name
            ]["nullable"]
        )

        observed_nullable = (
            live_columns[
                column_name
            ]["nullable"]
        )

        if (
            expected_nullable
            != observed_nullable
        ):
            nullability_mismatches[
                column_name
            ] = {
                "expected": (
                    expected_nullable
                ),
                "observed": (
                    observed_nullable
                ),
            }

    observed_primary_key = (
        fetch_primary_key_columns(
            cursor=cursor,
            schema_name=schema_name,
            table_name=table_name,
        )
    )

    expected_primary_key = (
        expected_contract[
            "primary_key"
        ]
    )

    primary_key_matches = (
        observed_primary_key
        == expected_primary_key
    )

    observed_constraints = (
        fetch_constraint_names(
            cursor=cursor,
            schema_name=schema_name,
            table_name=table_name,
        )
    )

    missing_constraints = sorted(
        expected_contract[
            "required_constraints"
        ]
        - observed_constraints
    )

    observed_indexes = (
        fetch_index_names(
            cursor=cursor,
            schema_name=schema_name,
            table_name=table_name,
        )
    )

    missing_indexes = sorted(
        expected_contract[
            "required_indexes"
        ]
        - observed_indexes
    )

    contract_valid = not any(
        (
            missing_columns,
            unexpected_columns,
            type_mismatches,
            nullability_mismatches,
            not primary_key_matches,
            missing_constraints,
            missing_indexes,
        )
    )

    return {
        "schema_name": schema_name,
        "table_name": table_name,
        "valid": contract_valid,
        "missing_columns": missing_columns,
        "unexpected_columns": (
            unexpected_columns
        ),
        "type_mismatches": (
            type_mismatches
        ),
        "nullability_mismatches": (
            nullability_mismatches
        ),
        "expected_primary_key": (
            expected_primary_key
        ),
        "observed_primary_key": (
            observed_primary_key
        ),
        "missing_constraints": (
            missing_constraints
        ),
        "missing_indexes": (
            missing_indexes
        ),
    }


# ------------------------------------------
# Provision and validate tables
# ------------------------------------------

connection = None
cursor = None

tables_existing_before = []
tables_created_this_run = []
table_validation_results = []


log_database_event(
    message=(
        "DATABASE_TABLE_PROVISIONING_START: "
        f"schema_version="
        f"{DATABASE_ACTIVE_SCHEMA_VERSION}; "
        f"quality_rules_version="
        f"{DATABASE_QUALITY_RULESET_VERSION}."
    ),
    level="INFO",
)


try:
    connection = psycopg2.connect(
        host=DB_HOST,
        dbname=DB_NAME,
        user=DB_USER,
        password=DB_PASSWORD,
        port=DB_PORT,
        sslmode=DB_SSLMODE,
        connect_timeout=10,
    )

    connection.autocommit = False
    cursor = connection.cursor()


    # --------------------------------------
    # Confirm required schemas still exist
    # --------------------------------------

    cursor.execute(
        """
        SELECT schema_name
        FROM information_schema.schemata
        WHERE schema_name = ANY(%s);
        """,
        (
            [
                "stg",
                "quarantine",
                "audit",
            ],
        ),
    )

    live_schema_names = {
        row[0]
        for row in cursor.fetchall()
    }

    missing_required_schemas = sorted(
        {
            "stg",
            "quarantine",
            "audit",
        }
        - live_schema_names
    )

    if missing_required_schemas:
        raise RuntimeError(
            "Required PostgreSQL schema(s) are missing: "
            + ", ".join(
                missing_required_schemas
            )
            + ". Rerun Part 4.1."
        )


    # --------------------------------------
    # Inspect tables before provisioning
    # --------------------------------------

    for schema_name, table_name in (
        EXPECTED_TABLE_CONTRACTS
    ):
        cursor.execute(
            """
            SELECT EXISTS (
                SELECT 1
                FROM information_schema.tables
                WHERE
                    table_schema = %s
                    AND table_name = %s
                    AND table_type = 'BASE TABLE'
            );
            """,
            (
                schema_name,
                table_name,
            ),
        )

        if bool(cursor.fetchone()[0]):
            tables_existing_before.append(
                f"{schema_name}.{table_name}"
            )


    # --------------------------------------
    # Create audit.pipeline_job_log
    # --------------------------------------

    cursor.execute(
        """
        CREATE TABLE IF NOT EXISTS
        audit.pipeline_job_log (
            job_id UUID NOT NULL,

            pipeline_session_id VARCHAR(64) NOT NULL,
            quality_audit_session_id VARCHAR(64) NOT NULL,

            source_file VARCHAR(255) NOT NULL,
            file_checksum VARCHAR(64) NOT NULL,

            status VARCHAR(30) NOT NULL,

            started_at TIMESTAMPTZ NOT NULL,
            completed_at TIMESTAMPTZ NULL,

            rows_read BIGINT NOT NULL DEFAULT 0,
            rows_loaded BIGINT NOT NULL DEFAULT 0,
            rows_quarantined BIGINT NOT NULL DEFAULT 0,

            duration_seconds NUMERIC(18, 3) NULL,

            schema_version VARCHAR(50) NOT NULL,
            quality_rules_version VARCHAR(50) NOT NULL,

            error_message TEXT NULL,

            created_at TIMESTAMPTZ NOT NULL
                DEFAULT CURRENT_TIMESTAMP,

            updated_at TIMESTAMPTZ NOT NULL
                DEFAULT CURRENT_TIMESTAMP,

            CONSTRAINT pk_pipeline_job_log
                PRIMARY KEY (job_id),

            CONSTRAINT ck_pipeline_job_log_status
                CHECK (
                    status IN (
                        'PENDING',
                        'RUNNING',
                        'SUCCESS',
                        'FAILED',
                        'SKIPPED'
                    )
                ),

            CONSTRAINT ck_pipeline_job_log_counts
                CHECK (
                    rows_read >= 0
                    AND rows_loaded >= 0
                    AND rows_quarantined >= 0
                )
        );
        """
    )


    # --------------------------------------
    # Create stg.cyclistic_rides
    # --------------------------------------

    cursor.execute(
        build_staging_table_statement()
    )


    # --------------------------------------
    # Create quarantine.cyclistic_rides
    # --------------------------------------

    cursor.execute(
        """
        CREATE TABLE IF NOT EXISTS
        quarantine.cyclistic_rides (
            quarantine_id UUID NOT NULL,

            ride_id VARCHAR(255) NULL,

            failed_rule VARCHAR(100) NOT NULL,
            reason TEXT NOT NULL,

            source_file VARCHAR(255) NOT NULL,
            source_row_number BIGINT NOT NULL,

            raw_record JSONB NOT NULL,

            pipeline_session_id VARCHAR(64) NOT NULL,
            quality_audit_session_id VARCHAR(64) NOT NULL,

            created_at TIMESTAMPTZ NOT NULL
                DEFAULT CURRENT_TIMESTAMP,

            CONSTRAINT pk_quarantine_cyclistic_rides
                PRIMARY KEY (quarantine_id)
        );
        """
    )


    # --------------------------------------
    # Record newly created tables
    # --------------------------------------

    all_expected_table_names = {
        f"{schema_name}.{table_name}"
        for schema_name, table_name
        in EXPECTED_TABLE_CONTRACTS
    }

    tables_created_this_run = sorted(
        all_expected_table_names
        - set(tables_existing_before)
    )


    # --------------------------------------
    # Create audit-table indexes
    # --------------------------------------

    cursor.execute(
        """
        CREATE UNIQUE INDEX IF NOT EXISTS
        ux_pipeline_job_log_success_file
        ON audit.pipeline_job_log (
            source_file,
            file_checksum
        )
        WHERE status = 'SUCCESS';
        """
    )

    cursor.execute(
        """
        CREATE INDEX IF NOT EXISTS
        ix_pipeline_job_log_session
        ON audit.pipeline_job_log (
            pipeline_session_id
        );
        """
    )

    cursor.execute(
        """
        CREATE INDEX IF NOT EXISTS
        ix_pipeline_job_log_source_file
        ON audit.pipeline_job_log (
            source_file
        );
        """
    )


    # --------------------------------------
    # Create staging indexes
    # --------------------------------------

    cursor.execute(
        """
        CREATE UNIQUE INDEX IF NOT EXISTS
        ux_stg_cyclistic_rides_ride_id
        ON stg.cyclistic_rides (
            ride_id
        );
        """
    )

    cursor.execute(
        """
        CREATE INDEX IF NOT EXISTS
        ix_stg_cyclistic_rides_source
        ON stg.cyclistic_rides (
            source_file,
            source_row_number
        );
        """
    )

    cursor.execute(
        """
        CREATE INDEX IF NOT EXISTS
        ix_stg_cyclistic_rides_session
        ON stg.cyclistic_rides (
            pipeline_session_id
        );
        """
    )


    # --------------------------------------
    # Create quarantine indexes
    # --------------------------------------

    cursor.execute(
        """
        CREATE INDEX IF NOT EXISTS
        ix_quarantine_cyclistic_rides_source
        ON quarantine.cyclistic_rides (
            source_file,
            source_row_number
        );
        """
    )

    cursor.execute(
        """
        CREATE INDEX IF NOT EXISTS
        ix_quarantine_cyclistic_rides_rule
        ON quarantine.cyclistic_rides (
            failed_rule
        );
        """
    )

    cursor.execute(
        """
        CREATE INDEX IF NOT EXISTS
        ix_quarantine_cyclistic_rides_session
        ON quarantine.cyclistic_rides (
            pipeline_session_id
        );
        """
    )


    # --------------------------------------
    # Validate all live table contracts
    # --------------------------------------

    for (
        schema_name,
        table_name,
    ), expected_contract in (
        EXPECTED_TABLE_CONTRACTS.items()
    ):
        validation_result = (
            validate_table_contract(
                cursor=cursor,
                schema_name=schema_name,
                table_name=table_name,
                expected_contract=(
                    expected_contract
                ),
            )
        )

        table_validation_results.append(
            validation_result
        )

        if not validation_result["valid"]:
            raise RuntimeError(
                "Database table contract mismatch for "
                f"{schema_name}.{table_name}: "
                f"{validation_result}"
            )

        log_database_event(
            message=(
                "DATABASE_TABLE_VERIFIED: "
                f"table={schema_name}.{table_name}; "
                f"columns="
                f"{len(expected_contract['columns'])}; "
                f"primary_key="
                f"{expected_contract['primary_key']}."
            ),
            level="INFO",
        )


    # --------------------------------------
    # Commit provisioning transaction
    # --------------------------------------

    connection.commit()

    log_database_event(
        message=(
            "DATABASE_TABLE_PROVISIONING_COMMITTED: "
            f"created="
            f"{len(tables_created_this_run)}; "
            f"existing="
            f"{len(tables_existing_before)}; "
            f"verified="
            f"{len(table_validation_results)}."
        ),
        level="INFO",
    )


except OperationalError as error:
    if connection is not None:
        connection.rollback()

    log_database_event(
        message=(
            "DATABASE_TABLE_PROVISIONING_FAILED: "
            "Unable to connect to PostgreSQL. "
            f"error={error}"
        ),
        level="CRITICAL",
    )

    raise ConnectionError(
        "Unable to open the PostgreSQL connection required "
        "for Part 4.2."
    ) from error


except (
    PermissionError,
    RuntimeError,
    ValueError,
    TypeError,
    psycopg2.DatabaseError,
) as error:
    if connection is not None:
        connection.rollback()

    log_database_event(
        message=(
            "DATABASE_TABLE_PROVISIONING_ROLLED_BACK: "
            f"error_type={type(error).__name__}; "
            f"error={error}"
        ),
        level="CRITICAL",
    )

    raise


finally:
    if cursor is not None:
        cursor.close()
        print("✓ Database cursor closed.")

    if connection is not None:
        connection.close()
        print("✓ Database connection closed.")


# ------------------------------------------
# Build Part 4.2 summary
# ------------------------------------------

DATABASE_TABLE_PROVISIONING_SUMMARY = {
    "database_processing_session_id": (
        DATABASE_PROCESSING_SESSION_ID
    ),
    "schema_version": (
        DATABASE_ACTIVE_SCHEMA_VERSION
    ),
    "quality_rules_version": (
        DATABASE_QUALITY_RULESET_VERSION
    ),
    "tables_existing_before": deepcopy(
        tables_existing_before
    ),
    "tables_created_this_run": deepcopy(
        tables_created_this_run
    ),
    "table_validation_results": deepcopy(
        table_validation_results
    ),
    "status": "SUCCESS",
    "completed_at": datetime.now(
        MELBOURNE_TIMEZONE
    ).isoformat(timespec="seconds"),
}


# ------------------------------------------
# Display Part 4.2 summary
# ------------------------------------------

print(
    "\n========== PostgreSQL Table Provisioning "
    "==========\n"
)

print(
    f"Database-processing session : "
    f"{DATABASE_PROCESSING_SESSION_ID}"
)
print(
    f"Approved schema version     : "
    f"{DATABASE_ACTIVE_SCHEMA_VERSION}"
)
print(
    f"Quality ruleset version     : "
    f"{DATABASE_QUALITY_RULESET_VERSION}"
)
print(
    f"Tables created              : "
    f"{len(tables_created_this_run)}"
)
print(
    f"Tables already existing     : "
    f"{len(tables_existing_before)}"
)
print(
    f"Tables verified             : "
    f"{len(table_validation_results)}"
)


print("\nTable details:")

for validation_result in (
    table_validation_results
):
    qualified_table_name = (
        f"{validation_result['schema_name']}."
        f"{validation_result['table_name']}"
    )

    print(
        f"  {qualified_table_name:<38} "
        f"columns="
        f"{len(EXPECTED_TABLE_CONTRACTS[
            (
                validation_result['schema_name'],
                validation_result['table_name'],
            )
        ]['columns']):<3} "
        f"status=VALID"
    )


print("\n" + "=" * 78)
print("✓ Required PostgreSQL tables exist.")
print("✓ Approved schema applied to the staging table.")
print("✓ Lineage columns provisioned.")
print("✓ Quarantine JSONB structure provisioned.")
print("✓ Job-log idempotency index provisioned.")
print("✓ Columns, keys, constraints, and indexes validated.")
print("✓ Part 4.2 completed successfully.")
print("✓ Ready for Part 4.3 load-session initialisation.")
print("=" * 78)